# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: **Refresh / Content Opportunity Scoring**. Data: the starter dataset that ships in this repo (`data/raw/content_refresh_anonymized.csv`) — no Hugging Face token needed for this notebook.

## 0. Load the data

In [9]:
import pandas as pd
import os

# Colab only opens this one file, not the whole repo, so pull the repo down first
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship'):
        !git clone -q https://github.com/heyzara124-hub/flyrank-ml-internship.git
    os.chdir('flyrank-ml-internship')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
# is_declining_label is added by the prep script normally; build it here the same way it's defined:
# trend_direction and trend_pct are the label's source, so they are NEVER used as features below.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df.shape)


(30000, 45)


## 1. My rule and its reason codes

**Two signal checks first**, one bucket table each, before I trust anything.

**Signal A — staleness, behind the refresh flags.** Does content that hasn't been updated in a long time decline more often?

In [10]:
staleness_check = df.groupby('freshness_tier')['is_declining_label'].agg(['mean', 'count'])
staleness_check.columns = ['decline_rate', 'n']
staleness_check


,decline_rate,n
freshness_tier,,
0-30,0.511377,20480
181+,0.471264,174
31-90,0.588571,175
91-180,0.611057,9171


**Verdict: MIXED.** Decline rate does rise from `0-30` (51.1%) to `91-180` (61.1%), which is the direction the refresh flags assume. But `31-90` and `181+` don't follow that line, and both have only ~175 rows each -- too few to trust on their own. The two big buckets (`0-30` and `91-180`, 20,480 and 9,171 rows) support the idea; the small buckets don't confirm or kill it. I'm not basing the rule on this signal alone.

**Signal B — CTR vs position, behind the CTR-fix logic.** Does CTR actually drop as position gets worse?

In [11]:
position_check = df.groupby('position_tier')['ctr'].agg(['mean', 'count'])
position_check.columns = ['avg_ctr', 'n']
position_check = position_check.reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
position_check


,avg_ctr,n
position_tier,,
top_3,1.483611,2321
page_1,0.652467,11814
striking,0.323239,7304
page_3_5,0.222484,7242
deep,0.150212,1319


**Verdict: CONFIRMED.** CTR steps down cleanly as position gets worse: `top_3` 1.48% -> `page_1` 0.65% -> `striking` 0.32% -> `page_3_5` 0.22% -> `deep` 0.15%. Every bucket has thousands of rows. This is the signal the CTR-fix logic assumes, and it holds.

**My rule, in plain words:** a page is worth reviewing for a CTR fix if its own CTR sits well below what pages at its position tier normally get, and it still gets enough traffic that the gap is real and not noise.

**Reason code:** `ctr_below_position_norm`
**Action label:** `review_meta_title_snippet`

## 2. Build the ranked queue (writes the CSV)

Score = 1 if CTR is under half the position tier's median CTR, AND impressions_90d >= 500 (the volume floor, so a 2-impression page can't top the list on a fluke). Score is scaled by impressions so bigger pages rank first among equally-underperforming ones.

In [12]:
median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')

underperforming = (df['ctr'] < 0.5 * median_ctr_by_tier).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)

df['score'] = underperforming * visible * df['impressions_90d']
df['reason_code'] = 'ctr_below_position_norm'
df['action_label'] = 'review_meta_title_snippet'

queue = df[df['score'] > 0].sort_values('score', ascending=False)
print(f'{len(queue):,} of {len(df):,} rows flagged for review')

import os
os.makedirs('work/outputs', exist_ok=True)
queue[['content_id', 'client_id', 'position_tier', 'ctr', 'impressions_90d',
       'score', 'reason_code', 'action_label']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False)
print('written to work/outputs/baseline_action_score.csv')
queue.head()


3,085 of 30,000 rows flagged for review
written to work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,score,reason_code,action_label
3394,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.48,0.0,excellent,page_1,stable,0.5,0,295097,ctr_below_position_norm,review_meta_title_snippet
6903,content_c84a0ab98e90,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2871.0,19536.0,...,25.00,0.0,excellent,page_1,stable,17.2,0,223271,ctr_below_position_norm,review_meta_title_snippet
7445,content_c8e9d6ab9013,client_19581e27de,20.0,0.03,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.0,excellent,page_1,down,-43.4,1,208678,ctr_below_position_norm,review_meta_title_snippet
3070,content_91652435f57a,client_19581e27de,10.0,0.29,LOW,0.09,keyword article,commercial,NaN,NaN,...,2.56,0.0,excellent,page_1,stable,-1.4,0,159590,ctr_below_position_norm,review_meta_title_snippet
5621,content_97a86caf3a3d,client_19581e27de,40.0,0.04,LOW,0.00,keyword article,transactional,NaN,NaN,...,9.21,0.0,excellent,page_1,down,-42.2,1,147670,ctr_below_position_norm,review_meta_title_snippet


## 3. Top-10 review

For each: the action, why it's there, and what would make it wrong.

In [13]:
top10 = queue.head(10)

for rank, (_, row) in enumerate(top10.iterrows(), 1):
    tier_med = median_ctr_by_tier[row.name]
    print(f"{rank}. action: review_meta_title_snippet")
    print(f"   why: ctr={row['ctr']:.2f}% is under half its {row['position_tier']} tier median "
          f"({tier_med:.2f}%), on {row['impressions_90d']:,} impressions -- real traffic, weak click-through")
    print(f"   wrong if: the low CTR is actually a mismatched search intent (wrong page for the "
          f"query) rather than a weak title/snippet, or if the impressions are mostly branded "
          f"queries where CTR is naturally low regardless of the snippet")
    print()


1. action: review_meta_title_snippet
   why: ctr=0.05% is under half its page_1 tier median (0.16%), on 295,097 impressions -- real traffic, weak click-through
   wrong if: the low CTR is actually a mismatched search intent (wrong page for the query) rather than a weak title/snippet, or if the impressions are mostly branded queries where CTR is naturally low regardless of the snippet

2. action: review_meta_title_snippet
   why: ctr=0.03% is under half its page_1 tier median (0.16%), on 223,271 impressions -- real traffic, weak click-through
   wrong if: the low CTR is actually a mismatched search intent (wrong page for the query) rather than a weak title/snippet, or if the impressions are mostly branded queries where CTR is naturally low regardless of the snippet

3. action: review_meta_title_snippet
   why: ctr=0.00% is under half its page_1 tier median (0.16%), on 208,678 impressions -- real traffic, weak click-through
   wrong if: the low CTR is actually a mismatched search intent 

## 4. Weak picks + leakage check

**Weak picks:** rows sitting right at the volume floor (impressions_90d close to 500) are the shakiest -- the underperforming/not-underperforming line can flip with one more or fewer click. Checking how many of the top 10 are close to that edge:

In [14]:
top10_margin = top10.assign(
    ctr_ratio=lambda d: d['ctr'] / median_ctr_by_tier.loc[d.index]
)[['content_id', 'position_tier', 'ctr', 'impressions_90d', 'ctr_ratio']]
print('ctr_ratio close to 0.5 = borderline call, not a clear underperformer')
top10_margin


ctr_ratio close to 0.5 = borderline call, not a clear underperformer


,content_id,position_tier,ctr,impressions_90d,ctr_ratio
3394,content_36ff89c8214e,page_1,0.05,295097,0.312500
6903,content_c84a0ab98e90,page_1,0.03,223271,0.187500
7445,content_c8e9d6ab9013,page_1,0.00,208678,0.000000
3070,content_91652435f57a,page_1,0.06,159590,0.375000
5621,content_97a86caf3a3d,page_1,0.07,147670,0.437500
27178,content_453722754fea,page_1,0.01,140079,0.062500
9193,content_c1fe78bc4e37,page_1,0.03,134055,0.187500
6689,content_e752a4e03dd3,page_3_5,0.01,130892,0.333333
3343,content_54baba704595,page_3_5,0.01,130617,0.333333
19183,content_124763d39ca5,page_3_5,0.01,129803,0.333333


None of the top 10 are borderline on CTR (all well under 0.3x their tier median), but all ten are large, high-traffic pages -- worth double-checking whether a handful of big clients are crowding out smaller opportunities that deserve a look too.

**Leakage check:** the rule only uses `ctr`, `position_tier`, and `impressions_90d` -- all measured over the same trailing 90-day window, none of them the future. `trend_direction`, `trend_pct`, and `is_declining_label` (the label source) are not used anywhere in the score. No product-decision fields (`health_score`, `priority_score`, etc.) exist in this file to begin with.

In [15]:
used_in_rule = {'ctr', 'position_tier', 'impressions_90d'}
banned = {'trend_direction', 'trend_pct', 'is_declining_label'}
print('rule inputs:', used_in_rule)
print('banned columns touched by the rule:', used_in_rule & banned)


rule inputs: {'ctr', 'position_tier', 'impressions_90d'}
banned columns touched by the rule: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.